# Song-History-Only MCQ Training

This notebook trains a multiple-choice model using only the song history JSONL data in this repo.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q -r requirements.txt

Create a deterministic train/validation/test split from `data/music_song_history_10mb.jsonl`. No other dataset is used.

In [ ]:
import json, random
from pathlib import Path

src = Path('data/music_song_history_10mb.jsonl')
out = Path('data/music_song_history_trainable.jsonl')
rows = [json.loads(line) for line in src.open(encoding='utf-8') if line.strip()]
rng = random.Random(7)
rng.shuffle(rows)
n = len(rows)
train_end = int(n * 0.88)
val_end = int(n * 0.94)
for i, row in enumerate(rows):
    row['split'] = 'train' if i < train_end else 'validation' if i < val_end else 'test'
with out.open('w', encoding='utf-8') as handle:
    for row in rows:
        handle.write(json.dumps(row, ensure_ascii=False) + '\n')
print({'rows': n, 'train': train_end, 'validation': val_end - train_end, 'test': n - val_end})

Train the song-history-only checkpoint. On Colab T4/L4/A100, CUDA should be used automatically.

In [ ]:
!python train_choice_classifier.py \
  --data data/music_song_history_trainable.jsonl \
  --base-model microsoft/deberta-v3-small \
  --out models/deberta_music_song_history_only_colab \
  --metrics-out models/deberta_music_song_history_only_colab_metrics.json \
  --epochs 2 \
  --batch-size 16 \
  --gradient-accumulation-steps 2 \
  --learning-rate 8e-6 \
  --max-length 160 \
  --freeze-lower-layers 0 \
  --eval-batch-size 32 \
  --log-every 100

Download or save the output folder `models/deberta_music_song_history_only_colab/` and the metrics JSON after training.

In [ ]:
!zip -r deberta_music_song_history_only_colab.zip models/deberta_music_song_history_only_colab models/deberta_music_song_history_only_colab_metrics.json